## **CUDA Runtime**

- Copies the data from the host to the device

- Load GPU kernel and execute it

- Copy the result from the device to the host

<hr>

## **`__global__`, `__device__`, and `__host__`**

Remember how we talked about `nvcc` acting as a "Compiler Driver" that cuts your code in half, sending the CPU code to `g++` and the GPU code to the PTX pipeline?

Because both your CPU code and your GPU code live inside the exact same `.cu` file, `nvcc` needs a way to know **which function belongs to which processor**. 

To solve this, CUDA introduces **Execution Space Specifiers**: `__host__`, `__global__`, and `__device__`. Think of these as nametags you put on your functions. They tell the compiler two very specific things:
1. **Where does this function run?** (CPU or GPU)
2. **Who is allowed to call this function?** (CPU or GPU)

Here is the in-depth breakdown of each, why they exist, and how they interact.

---

### 1. `__host__` (The Standard CPU Function)

* **Executes on:** The Host (CPU)

* **Callable from:** The Host (CPU) ONLY, we can't call from the GPU Kernel

**What it is:**

This is just a normal, everyday C++ function. In fact, if you write a function and don't put *any* tag on it, the compiler automatically assumes it is `__host__`. 

**Why it exists:**

You almost never have to type `__host__` explicitly. It exists mostly so you can combine it with `__device__` (which I will show you in the "Secret Combo" below!).

```cpp
// You don't need to type __host__, but this is exactly what it means!
__host__ void printHelloFromCPU() {
    printf("I am running on the AMD Ryzen CPU!\n");
}

int main() {
    printHelloFromCPU(); // CPU calls a CPU function. Perfectly fine.
    return 0;
}
```

---

### 2. `__global__` (The Bridge / The Kernel)

* **Executes on:** The Device (GPU)

* **Callable from:** The Host (CPU)*, but with a special syntax: `<<<blocks, threads>>>`. Executed by thousands of GPU threads in parallel.

**What it is:**

The `__global__` tag is the most important word in CUDA. It defines a **Kernel**. This is the main door, or the "bridge," between your CPU and your GPU. It allows your CPU to order the GPU to start doing work.

**Strict Rules for `__global__`:**

1. It **must** return `void`. A GPU kernel cannot use `return 5;` back to the CPU, because the CPU doesn't wait for the GPU to finish (it is asynchronous). To get data back, you must write it to a pointer in VRAM and use `cudaMemcpy`.

2. Whenever the CPU calls a `__global__` function, it **must** use the execution configuration syntax: `<<<blocks, threads>>>`.

```cpp
// Executes on the RTX 3060, but is triggered by the Ryzen CPU
__global__ void addMathKernel(int* d_array) {
    int tid = threadIdx.x;
    d_array[tid] = d_array[tid] + 10;
}

int main() {
    // The CPU stands on the edge of the bridge and shouts the command over to the GPU
    addMathKernel<<<1, 256>>>(d_array); 
    return 0;
}
```

*(Fun Advanced Fact: Since CUDA 5.0, NVIDIA introduced "Dynamic Parallelism", which actually allows a `__global__` function to call another `__global__` function right from the GPU, but as a beginner, just think of it as the CPU-to-GPU bridge!)*

---

### 3. `__device__` (The GPU Helper Function)

* **Executes on:** The Device (GPU)

* **Callable from:** The Device (GPU) ONLY

**What it is:**

As your GPU programs get extremely complex, you don't want to write a 1,000-line `__global__` kernel. You want to break your code up into smaller, modular functions just like you do in normal C++. 

A `__device__` function is a helper function that lives entirely inside the GPU. 

* The CPU cannot see it.

* The CPU cannot call it.

* It can only be called by a `__global__` kernel, or by another `__device__` function.

**Unlike `__global__`, a `__device__` function CAN return values!**

```cpp
// 1. The Helper Function (GPU Only)
__device__ int squareNumber(int a) {
    return a * a; // It can return values!
}

// 2. The Main Kernel (GPU execution, CPU trigger)
__global__ void processArray(int* d_array) {
    int tid = threadIdx.x;
    
    // The GPU kernel calls its own internal GPU helper function
    d_array[tid] = squareNumber(d_array[tid]); 
}

int main() {
    processArray<<<1, 256>>>(d_array);
    
    // squareNumber(5); // COMPILER ERROR! The CPU cannot reach a __device__ function.
    return 0;
}
```

---

### 4. The Secret Combo: `__host__ __device__` 

Sometimes, you write a fantastic math function (like calculating the distance between two 3D points). 

You want to use it in your CPU code to test a single point, but you also want to use it inside your GPU kernel to process a million points.

If you tag it as `__host__`, the GPU can't use it.
If you tag it as `__device__`, the CPU can't use it.
If you don't tag it at all, `nvcc` assumes it's `__host__`.

**The Solution:** You give it **both tags**.

```cpp
__host__ __device__ float getDistance(float x, float y) {
    return sqrt((x * x) + (y * y));
}
```
**What happens under the hood:**

When `nvcc` sees both tags, it literally compiles **two entirely separate versions** of the exact same function. It builds an x86 machine-code version for your Ryzen CPU, and a `PTX (Parallel Thread Execution)/SASS` version for your RTX 3060. Now, both your `main()` function and your `__global__` kernels can call `getDistance()` flawlessly!

> PTX (Parallel Thread Execution). It is a low-level, hardware-agnostic Intermediate Representation (IR) and virtual Instruction Set Architecture (ISA) used for writing and optimizing GPU code. It is an intermediate language that sits between high-level CUDA C++ code and the final machine code (SASS) that runs on NVIDIA GPUs. It has `.ptx` files, which are human-readable assembly-like code that can be further compiled into SASS (the actual machine code for the GPU).

> SASS (Streaming Assembler). It is the final machine code that runs directly on NVIDIA GPUs. It is generated from PTX code by the NVIDIA driver at runtime. `SASS` is specific to the architecture of the GPU (e.g., Volta, Turing, Ampere) and is not human-readable. It is optimized for performance and efficiency on the target GPU. The `PTX` is compiled into `SASS` (`.cubin` files) by the NVIDIA driver when the program is executed.

---

### Putting it all together (The Master Example)

Here is a complete, single program that shows exactly how all three interact in a real CUDA codebase. Read the comments to see who is calling who.

```cpp
#include <iostream>
#include <stdio.h>

// ---------------------------------------------------------
// 1. HOST-ONLY: Normal C++ function
// ---------------------------------------------------------
__host__ void printIntro() {
    printf("Starting the program on the CPU...\n");
}

// ---------------------------------------------------------
// 2. DEVICE-ONLY: Helper function for the GPU
// ---------------------------------------------------------
__device__ int doubleTheValue(int val) {
    return val * 2;
}

// ---------------------------------------------------------
// 3. HOST & DEVICE: Can be called by anyone!
// ---------------------------------------------------------
__host__ __device__ int addFive(int val) {
    return val + 5;
}

// ---------------------------------------------------------
// 4. GLOBAL: The Bridge. Runs on GPU, called by CPU.
// ---------------------------------------------------------
__global__ void myKernel(int* d_data) {
    int tid = threadIdx.x;
    
    // Kernel calls a __device__ function
    int doubled = doubleTheValue(d_data[tid]); 
    
    // Kernel calls a __host__ __device__ function
    d_data[tid] = addFive(doubled); 
}

// ---------------------------------------------------------
// 5. MAIN: The starting point on the CPU
// ---------------------------------------------------------
int main() {
    // 1. CPU calls a __host__ function
    printIntro(); 

    // 2. CPU calls a __host__ __device__ function (Just to test it!)
    int cpu_test = addFive(10); 
    printf("CPU tested addFive(10) and got: %d\n", cpu_test);

    // ... (Imagine cudaMalloc and cudaMemcpy happen here) ...
    int* d_data;

    // 3. CPU calls a __global__ function to wake up the GPU
    // myKernel<<<1, 32>>>(d_data);

    // 4. CPU waits for GPU to finish
    // cudaDeviceSynchronize();

    return 0;
}
```

<hr>

## **Memory Management**

We just cannot directly use our CPU's RAM from the GPU. The GPU has its own separate memory called VRAM. At first everything is in the `RAM` of the CPU, and if we want to use it on the GPU, we have to explicitly copy it over to the VRAM. The GPU will use VRAM to do all its processing, and then if we want to get the results back to the CPU, we have to copy it back from VRAM to RAM.

### 1. `cudaMalloc`

**What it does:**

It reserves a contiguous block of memory inside the **Global Memory (VRAM)** of your RTX 3060. 

**How the syntax works:**

```cpp
float *d_a; 
cudaMalloc(&d_a, N*N*sizeof(float));
```
* **`N*N`**: This instantly tells us we are dealing with a flattened 2D grid! (Like an image with `N` width and `N` height).

* **The Pointer Paradox (`&d_a`)**: Notice that `float *d_a;` is created on the CPU. It is a CPU variable. But we want it to hold a GPU memory address. We pass `&d_a` (the address of the pointer) to `cudaMalloc` so the NVIDIA driver can physically modify our CPU pointer and inject the brand-new GPU memory address into it. 

**Why it is called "Global Memory":**

The VRAM on your graphics card is called "Global" because **everyone can see it**. The CPU can see it (via `cudaMemcpy`), and every single one of the thousands of GPU threads can read and write to it simultaneously.

---

### 2. `cudaMemcpy`

**What it does:**

It physically moves bytes of data across the PCIe bus on your motherboard. 

**Why we need directions:**

The hardware needs to know which way the traffic is flowing to optimize the DMA (Direct Memory Access) controllers. 

* **`cudaMemcpyHostToDevice` (CPU to GPU):** Used at the start of your program. You load a file from your SSD into CPU RAM, and then ship it to the GPU to be processed.

* **`cudaMemcpyDeviceToHost` (GPU to CPU):** Used at the end of your program. The GPU cannot save files to your SSD. It must ship the finished math back to the CPU so your C++ program can save the file.

* **`cudaMemcpyDeviceToDevice` (GPU to GPU):** Used purely inside the VRAM. **Why do this?** Imagine you are blurring an image. You need to read the original pixels to calculate the blur, but write the blurred pixels to a new location so you don't corrupt the original data mid-calculation. Copying from one VRAM location to another VRAM location operates at roughly **300+ GB/s**, whereas copying over the PCIe bus is only about **16 GB/s**.

---

### 3. `cudaFree

**What it does:**

It releases the VRAM back to the NVIDIA driver.

**Why it is critical:**

Your RTX 3060 Laptop GPU has exactly **6 GB of VRAM**. 

If you process a video frame that takes 100 MB of VRAM, and you forget to call `cudaFree` at the end of your loop, your GPU will "leak" 100 MB per frame.

At 60 frames per second, **you will completely fill and crash your graphics card in exactly 1 second.** `cudaFree` is non-negotiable!

---

### The Real-World Example: 4K Image Processing

Let’s step out of abstract "Vector A and Vector B" and look at a real-world scenario. 
Imagine we are writing a Photoshop-style application. We want to take a massive 4K high-dynamic-range (HDR) image and apply a Brightness Filter to it.

HDR image pixels are stored as `float` values (decimals). 
A 4K image is roughly `4000 x 4000` pixels (`N * N`).

Here is the exact memory management pipeline to process that image.

```cpp
#include <iostream>
#include <vector>

int main() {
    // 1. Define our image size
    int N = 4000; // 4000 width x 4000 height
    size_t num_pixels = N * N; // 16,000,000 pixels
    size_t bytes = num_pixels * sizeof(float); // ~64 Megabytes of data

    // ==========================================
    // CPU PHASE: Load the Image
    // ==========================================
    // Create the image array in CPU RAM
    std::vector<float> h_image_in(num_pixels, 0.5f); // Pretend we loaded a gray image
    std::vector<float> h_image_out(num_pixels);      // Empty array to hold the result

    // ==========================================
    // GPU LOGISTICS PHASE 1: Allocate VRAM
    // ==========================================
    float *d_image_in, *d_image_out;

    // We need 64 MB for the original image, and 64 MB for the new brightened image
    cudaMalloc(&d_image_in, bytes);
    cudaMalloc(&d_image_out, bytes);

    // ==========================================
    // GPU LOGISTICS PHASE 2: Ship the Data
    // ==========================================
    std::cout << "Shipping 64MB Image to GPU...\n";
    // Send the raw image from Host (CPU) to Device (GPU)
    cudaMemcpy(d_image_in, h_image_in.data(), bytes, cudaMemcpyHostToDevice);

    // ==========================================
    // COMPUTE PHASE: The Magic Happens
    // ==========================================
    // (Imagine we launch a kernel here that adds +0.2f brightness to every pixel)
    // brightenImageKernel<<<Blocks, Threads>>>(d_image_in, d_image_out, num_pixels);
    
    // Here is a Device-to-Device example! 
    // Let's pretend the kernel failed, and we just want to copy the original 
    // image straight into the output buffer entirely within the GPU VRAM:
    cudaMemcpy(d_image_out, d_image_in, bytes, cudaMemcpyDeviceToDevice);

    // ==========================================
    // GPU LOGISTICS PHASE 3: Retrieve the Product
    // ==========================================
    std::cout << "Retrieving finished Image from GPU...\n";
    // Send the finished, brightened image from Device (GPU) back to Host (CPU)
    cudaMemcpy(h_image_out.data(), d_image_out, bytes, cudaMemcpyDeviceToHost);

    // ==========================================
    // GPU LOGISTICS PHASE 4: Clean Up
    // ==========================================
    // We saved our result to the CPU, so the GPU doesn't need the memory anymore.
    cudaFree(d_image_in);
    cudaFree(d_image_out);

    std::cout << "VRAM Freed. Image ready to save to hard drive!\n";

    return 0;
}
```

<hr>

What you have just outlined is the complete **Compilation Pipeline** of CUDA. This is one of the most brilliant engineering feats by NVIDIA, and understanding it explains exactly why CUDA dominates the high-performance computing world.

To understand this, you must first know one secret: **`nvcc` is not actually a compiler.** 

`nvcc` (NVIDIA CUDA Compiler) is a **Compiler Driver**. It is an orchestrator. When you feed it your `vadd.cu` file, `nvcc` takes out a scalpel, cuts your code perfectly in half (CPU vs. GPU), and sends each half down a completely different pipeline. 

Here is the in-depth breakdown of exactly what happens.

---

### Phase 1: The Host Code (CPU) Pipeline

Your CPU (`g++` / Ryzen processor) and your GPU (`nvcc` / RTX 3060) speak completely different languages. A standard C++ compiler like `g++` has absolutely no idea what `__global__` or `<<<1, 256>>>` means. If it sees them, it throws a syntax error.

**1. "Modified to run kernels"**
Before passing the code to your CPU compiler, `nvcc` performs a translation. It strips out all the GPU kernel code. Then, it looks at your kernel launch:
`vectorAdd<<<NUM_BLOCKS, NUM_THREADS>>>(d_a, d_b, d_c, n);`

It rewrites this weird `<<< >>>` syntax into standard, ugly C++ API functions. It transforms it into something like:
`cudaLaunchKernel((void*)vectorAdd, NUM_BLOCKS, NUM_THREADS, args);`

**2. "Compiled to x86 binary"**
Now that the file is 100% standard, pure C++, `nvcc` hands it over to your system's host compiler (in your case, `g++ 16.1`). 
`g++` compiles this code into standard **x86_64 machine code** that your AMD Ryzen 5 CPU can execute natively.

---

### Phase 2: The Device Code (GPU) Pipeline

While `g++` is busy compiling the CPU code, `nvcc` takes the `__global__` functions and starts compiling the GPU code. But it doesn't compile it to raw 1s and 0s immediately. 

**1. "Compiled to PTX"**
Instead of creating actual machine code, `nvcc` compiles your GPU code into **PTX (Parallel Thread Execution)**. 

PTX is an "Intermediate Representation." It is a fake, virtual assembly language. It acts as if it is compiling for a "perfect, theoretical" NVIDIA GPU with infinite registers and infinite memory. It is plain text, and you can actually read it if you want to!

**2. "Stable across multiple GPU generations"**
Why a fake assembly language? Because GPU hardware changes violently every 2 years. 
The physical micro-architecture of a 2016 GTX 1060 (Pascal) is wildly different from your 2021 RTX 3060 (Ampere). If NVIDIA forced you to write specific code for every single graphics card, developers would quit. 
PTX solves this. PTX is a universal, stable language. NVIDIA ensures that PTX generated 10 years ago is completely understandable by the NVIDIA driver of today.

---

### Phase 3: Execution and JIT (Just-In-Time) Compilation

So, you have your final executable file (like `./vadd`). Inside this executable is your compiled CPU code, and the text-based PTX blueprints for your GPU code. 

You press `Enter` to run the program.

**1. "PTX into native GPU instructions"**
Your physical RTX 3060 cannot read PTX. It only understands raw, physical microcode called **SASS** (Streaming ASSembler) or **cubin** (CUDA Binary). 

When your program reaches the `cudaLaunchKernel` step, the NVIDIA Display Driver on your computer wakes up. The driver looks at the PTX blueprints hidden in the executable, looks at your physical hardware, and says: *"Ah! You have an Ampere RTX 3060!"*

In a fraction of a millisecond, the driver performs **JIT (Just-In-Time) Compilation**. It translates the generic PTX text into the exact, highly-optimized, physical SASS machine code tailored perfectly for your specific GPU's silicon, and feeds it to the graphics card.

**2. "Allows for forward compatibility"**
This is the ultimate superpower of CUDA. 

Imagine you write your `vadd.cu` code today, compile it to an executable, and upload it to the internet.
Five years from now, someone downloads your executable and tries to run it on an **RTX 6090** (an architecture that hasn't even been invented yet). 

* If you had compiled it directly to RTX 3060 microcode, the program would crash. The RTX 6090 wouldn't understand the old instructions.
* **But because you embedded PTX**, the program runs perfectly. The future RTX 6090 driver will look at the PTX, JIT-compile it into RTX 6090 SASS, and run it flawlessly. 

You wrote code that runs on hardware from the future.

### Summary / How this affects your compilation flags:
Remember earlier when I told you to use `nvcc -arch=sm_86`? 
* `sm_86` is the physical architecture of your RTX 3060. 
* By using that flag, you told `nvcc`: *"Skip the JIT part for my machine! Compile it directly to physical SASS microcode for the RTX 3060 so it launches instantly!"* (This is called Ahead-Of-Time or AOT compilation).

If you were building software to sell on Steam to millions of gamers with different GPUs, you would use flags that embed the generic PTX so their individual drivers could JIT-compile it!